# Count mean specification

A simpler mean specification can reduce count prediction error when a smooth draft expends updates estimating an approximately linear log mean. This study compares a linear-backbone candidate with the previous discovery procedure on five count processes and additional nonlinear, null, noise and genuine-interaction controls. The [protocol](count_mean_backbone/PROTOCOL.md), [interaction control](count_mean_backbone/INTERACTION_CONTROL_ADDENDUM.md), [scope](count_mean_backbone/COUNT_BRANCH_SCOPE.md) and [candidate decision](count_mean_backbone/FINAL_CANDIDATE_DECISION.md) precede fresh confirmation.

Each process has ten development and fifty fresh confirmation draws, each with 1,000 analysis rows, a shared 500/500 fitting/calibration partition and 1,000 independent evaluation rows. The two information tracks either discover a specification from fitting rows or supply a mean formula and tune it on fitting rows. Calibration and evaluation responses select neither candidate nor settings. Nominal coverage is 0.90; efficiency comparisons require both methods' mean coverage in [0.88,0.92]. The primary width and integer-cardinality targets use one-sided 95% upper paired excess bounds of at most 1% against prespecified LightGBM CQR. Development draws remain separate from confirmation.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import polars as pl
from great_tables import GT, md

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "Cargo.toml").exists())
sys.path.insert(0, str(root / "python"))
sys.path.insert(0, str(root))
from benches.evidence_float_codec import decode_record

directory = root / "docs/book/benchmarks/count_mean_backbone"
summary = decode_record(json.loads((directory / "production_confirmation_summary.json").read_text()))
rows = summary["rows"]
gate = json.loads((directory / "production_confirmation_gate.json").read_text())
labels = {"baseline_discovery": "Prior discovery", "rate_discovery": "Rate-only control",
          "shape_discovery": "Linear-backbone candidate", "combined_discovery": "Shape + rate control",
          "supplied_fixed": "Supplied / 100 updates", "supplied_tuned": "Supplied / tuned"}

In [ ]:
def comparison_table(records, title):
    leaders = {}
    for panel in {r["panel"] for r in records}:
        eligible = [r for r in records if r["panel"] == panel and r.get("coverage_eligible")
                    and not r["failures"] and not r["unbounded"]]
        for key in ("width", "count_size", "true_mean_mse", "interval_score"):
            leaders[panel, key] = min((r[key] for r in eligible), default=None)

    def value(row, key):
        if key not in row:
            return "unavailable"
        number = row[key]
        text = f"{number:.3f}"
        if key in ("width", "count_size", "true_mean_mse", "point_mse", "interval_score") and row.get(key + "_se") is not None:
            text += f" ± {row[key + '_se']:.3f}"
        return f"**{text}**" if row.get("coverage_eligible") and number == leaders.get((row["panel"], key)) else text

    frame = pl.DataFrame([{"Process": r["panel"], "Method": labels.get(r["method"], r["method"].replace("_", " ")),
        "Width": value(r, "width"), "Count size": value(r, "count_size"), "Coverage": value(r, "coverage"),
        "Mean MSE": value(r, "true_mean_mse"), "Response MSE": value(r, "point_mse"), "Interval score": value(r, "interval_score"),
        "Failed": r["failures"], "Unbounded": r["unbounded"]} for r in records])
    return (GT(frame, rowname_col="Method", groupname_col="Process").tab_stubhead("Workflow")
        .tab_header(title=title, subtitle="50 draws per process; smaller size, width, MSE and interval score are better")
        .tab_spanner(label="Prediction sets", columns=["Width", "Count size", "Coverage", "Interval score"])
        .fmt_markdown(columns=["Width", "Count size", "Mean MSE", "Interval score"])
        .tab_source_note(md("Width is real length in count-response units; count size is the number of nonnegative integers "
          "contained. Size, width and error summaries show mean ± Monte Carlo SE across independent draws. Mean MSE uses the generating conditional "
          "mean, in squared count units. CQR point predictions are band midpoints. Bold marks minima among complete "
          "coverage-eligible rows. Failed methods have no successful-only mean; all unbounded outcomes are retained. "
          "Source: production_confirmation_summary.json; runner benches/study_count_mean_backbone.py.")))

In [ ]:
headline = [r for r in rows if r["panel"] in ("original", "zero_inflated")]
comparison_table(headline, "Every method on the primary count processes")

Fixed CatBoost split/CQR controls use 500 trees, depth 6 and learning rate 0.05; LightGBM uses 300 trees, 15 leaves and rate 0.05; sklearn CQR uses 300 trees, depth 3 and rate 0.05. Every competitor uses the draw seed and one thread. Supplied Poisson controls use either 100 fixed updates at rate 0.1 or five-fold stopping selection starting at 100, capped at 400, at rate 0.02. Supplied mean formulas are `bols(x1)+bols(x2)`, `bbs(x1,df=4)+bols(x2)` for nonlinear controls, `1` for nulls, and `bols(x1)+bols(x2)+bols(x1,by=x2,intercept=False)` for the genuine interaction. Their supplied information remains separate from discovery.

## Mean-form comparison

Before backward elimination, the candidate compares the autogenerated Poisson or negative-binomial smooth draft with a linear backbone on the same variables. On five fitting folds at 200 updates, it keeps the simpler specification when its mean family-risk increase is at most twice the standard error of paired fold differences. This is a declared complexity heuristic. Exact equality prefers the simpler specification; there is no numerical tie tolerance. Mandatory terms, declared transformations, other learner types and cost admission are preserved.

The existing learning rates remain 0.1 for Gaussian and 0.02 for other families. Rate-only and combined variants remain diagnostic controls. They cannot replace the primary candidate after confirmation. Gaussian mean and scale selection rules are unchanged. Numeric shape-choice and learning-rate lookup use the shared Rust core with Python and R bindings.

In [ ]:
targets = [r for r in rows if (r["panel"], r["method"]) in {
    ("original", "shape_discovery"), ("original", "supplied_fixed"),
    ("zero_inflated", "shape_discovery"), ("zero_inflated", "supplied_tuned")}]
frame = pl.DataFrame([{"Process": r["panel"], "Workflow": labels[r["method"]], "Coverage": r.get("coverage"),
    "Width excess %": r.get("width_excess"), "Width upper %": r.get("width_upper95"),
    "Count excess %": r.get("count_size_excess"), "Count upper %": r.get("count_size_upper95"),
    "Both targets": r.get("width_upper_target", False) and r.get("count_size_upper_target", False)} for r in targets])
(GT(frame, rowname_col="Workflow", groupname_col="Process").tab_stubhead("Information track")
 .tab_header(title="Frozen one-percent targets", subtitle="Paired percent excess over LightGBM CQR; upper limits ≤ 1% required")
 .fmt_number(columns=["Coverage", "Width excess %", "Width upper %", "Count excess %", "Count upper %"], decimals=3)
 .tab_source_note("Each draw contributes 100 × (method size / reference size − 1), separately for real width and integer "
   "cardinality. One-sided 95% Student-t upper limits use 49 degrees of freedom. Both mean coverages must be in [0.88,0.92]."))

In [ ]:
comparisons = [{"Process": r["panel"], "Workflow": labels[r["method"]], "Control": c["control"],
    "Eligible control": c["eligible"], "Width upper %": c.get("width_upper_adjusted95"),
    "Count upper %": c.get("count_size_upper_adjusted95")} for r in targets for c in r.get("comparisons", [])]
(GT(pl.DataFrame(comparisons), rowname_col="Control", groupname_col="Process").tab_stubhead("Fixed competitor")
 .tab_header(title="All five competitor contrasts", subtitle="Bonferroni-adjusted one-sided upper paired excess, in percentage points")
 .fmt_number(columns=["Width upper %", "Count upper %"], decimals=3)
 .tab_source_note("The adjusted bound uses Student-t(0.99,49) for five comparisons. Control eligibility uses [0.88,0.92] "
   "coverage. These separately reported bounds do not replace the frozen prespecified LightGBM comparison."))

In [ ]:
decisions = [{"Requirement": "Primary width and count targets", "Passed": gate["primary_width_count_upper_targets"]},
             {"Requirement": "Candidate finite and coverage eligible", "Passed": gate["candidate_finite_and_eligible"]},
             {"Requirement": "Null validity", "Passed": gate["null_validity"]}]
decisions += [{"Requirement": r["panel"] + " / nonlinear or interaction control", "Passed": r.get("pass", False)}
              for r in gate["regression_controls"]]
decisions += [{"Requirement": r["panel"] + " / false inclusion", "Passed": r.get("pass_", False)}
              for r in gate["false_inclusion_checks"]]
decisions += [{"Requirement": "Complete frozen promotion gate", "Passed": gate["promotion_pass"]}]
(GT(pl.DataFrame(decisions), rowname_col="Requirement").tab_stubhead("Requirement")
 .tab_header(title="Complete confirmation decision", subtitle="Every requirement was specified before fresh draws")
 .tab_source_note("Curved terms must remain in ≥90% of nonlinear draws. The family-supplied genuine-interaction control "
   "requires ≥90% interaction retention. Paired latent-mean MSE upper excess must be ≤10%. The ordinary interaction control "
   "requires non-regression; it does not establish the unmet unconditional 90% aspiration. Null/noise false inclusion uses "
   "a one-sided exact paired McNemar test at 0.05. No failing requirement is replaced by a favorable diagnostic control."))

In [ ]:
lookup = {(r["panel"], r["method"]): r for r in rows}
zi_candidate, zi_previous = (lookup["zero_inflated", method] for method in ("shape_discovery", "baseline_discovery"))
zi_fixed, zi_tuned = (lookup["zero_inflated", method] for method in ("supplied_fixed", "supplied_tuned"))
ordinary_interaction = lookup["interaction_zero_inflated", "shape_discovery"]
hinted_interaction = lookup["interaction_supplied_family", "shape_discovery"]
hinted_check = next(r for r in gate["regression_controls"] if r["panel"] == "interaction_supplied_family")

The candidate passes every frozen requirement. On zero-inflated counts, its count-size upper excess bound is {eval}`f"{zi_candidate['count_size_upper95']:+.3f}%"`, and its real-width upper bound is {eval}`f"{zi_candidate['width_upper95']:.3f}%"`, with coverage {eval}`f"{zi_candidate['coverage']:.5f}"`. The tuned supplied count-size upper bound is {eval}`f"{zi_tuned['count_size_upper95']:.3f}%"`. On original-rate counts, candidate and fixed supplied width/cardinality targets also pass. These are bounded confirmation results for the stated processes and information tracks.

The previous discovery procedure also passes the zero-inflated target on this fresh cohort: its count-size upper bound is {eval}`f"{zi_previous['count_size_upper95']:+.3f}%"`. Consequently this comparison does not establish that the new complexity choice is necessary to cross the target. Candidate latent-mean MSE is {eval}`f"{zi_candidate['true_mean_mse']:.3f}"` versus {eval}`f"{zi_previous['true_mean_mse']:.3f}"` for the prior procedure, in squared count units; the table reports Monte Carlo uncertainty. The old fixed supplied zero-inflated preset remains inefficient, with count-size upper excess {eval}`f"{zi_fixed['count_size_upper95']:+.3f}%"`.

The ordinary genuine-interaction procedure retains interactions on {eval}`f"{ordinary_interaction['interactions']}/{ordinary_interaction['draws']}"` draws, with unchanged latent-mean MSE {eval}`f"{ordinary_interaction['true_mean_mse']:.3f}"`. This falls short of the original unconditional 90% representation aspiration. The explicitly family-supplied diagnostic retains {eval}`f"{hinted_interaction['interactions']}/{hinted_interaction['draws']}"` and has candidate MSE {eval}`f"{hinted_interaction['true_mean_mse']:.3f}"`; its upper paired MSE excess over its own baseline is {eval}`f"{hinted_check['mse_excess_upper95']:+.3f}%"`. These separate results support the scoped non-regression check, and do not establish that ordinary discovery recovers the correct count family or mean on this interaction process.

## Nonlinear and interaction controls

The five original processes have mean $a\exp(0.3+0.6x_1-0.4x_2)$: low, original and high Poisson rates use $a=0.2,1,10$; negative-binomial and zero-inflated panels also use $a=10$. The negative-binomial shape is 5, giving variance $\mu+\mu^2/5$. Added controls use $a=10$. Nonlinear controls replace $0.6x_1$ with $0.8\sin(\pi x_1)$. Null controls have constant mean $10\exp(0.3)$. A noise control adds two independent irrelevant predictors. Predictors are independent Uniform(−1,1). Zero-inflated controls draw a zero with probability 0.3, otherwise Poisson with mean equal to the marginal mean divided by 0.7.

The genuine-interaction mean is $10\exp(0.3+0.6x_1-0.4x_2+1.2x_1x_2)$. Ordinary discovery chooses its family. A separately labelled diagnostic supplies only the negative-binomial family on the same observations, to exercise the changed count branch. This diagnostic is not data-only evidence and is not an independent second sample. The nonlinear additive limitation in [#679](https://github.com/fcela/mboostrs/issues/679) and sharp-mean scale attribution in [#629](https://github.com/fcela/mboostrs/issues/629) remain separate.

In [ ]:
controls = [r for r in rows if r["method"] in ("baseline_discovery", "shape_discovery")
            and r["panel"] not in ("original", "zero_inflated")]
comparison_table(controls, "Prior and candidate discovery on every control")

In [ ]:
selection = [r for r in rows if r["method"] in ("baseline_discovery", "shape_discovery") and not r["failures"]]
frame = pl.DataFrame([{"Process": r["panel"], "Workflow": labels[r["method"]],
    "Curved x1": r["curve_terms"], "Interactions": r["interactions"], "Nonconstant": r["nonconstant"],
    "Noise selected": r["noise_inclusions"], "Low-scale coverage": r["coverage_low"],
    "High-scale coverage": r["coverage_high"]} for r in selection])
(GT(frame, rowname_col="Workflow", groupname_col="Process").tab_stubhead("Procedure")
 .tab_header(title="Specification retention and conditional coverage", subtitle="Selection counts out of 50 draws; target coverage 0.90")
 .fmt_number(columns=["Low-scale coverage", "High-scale coverage"], decimals=3)
 .tab_source_note("Curved x1 counts formulas containing bbs(x1); interaction counts tensor, varying-coefficient or tree terms. "
   "These representation counts are descriptive, not proofs of recovered structure. Conditional coverages use the lowest "
   "and highest thirds of the generating standard deviation; tied constant scales include all rows. Null coverage above "
   "0.92 remains ineligible for an efficiency claim."))

In [ ]:
from scipy.stats import binomtest

inclusion = []
for r in rows:
    if r["method"] not in ("baseline_discovery", "shape_discovery") or r["failures"]:
        continue
    if r["panel"] not in ("null_poisson", "null_zero_inflated", "noise_zero_inflated"):
        continue
    count = r["noise_inclusions"] if r["panel"].startswith("noise") else r["nonconstant"]
    interval = binomtest(count, r["draws"]).proportion_ci(confidence_level=.95, method="exact")
    inclusion.append({"Process": r["panel"], "Workflow": labels[r["method"]], "Selected": count,
                      "Lower probability": interval.low, "Upper probability": interval.high})
(GT(pl.DataFrame(inclusion), rowname_col="Workflow", groupname_col="Process").tab_stubhead("Procedure")
 .tab_header(title="False inclusion uncertainty", subtitle="Counts out of 50 independent draws; exact binomial 95% intervals")
 .fmt_number(columns=["Lower probability", "Upper probability"], decimals=3)
 .tab_source_note("Null panels count any nonconstant mean; the noise panel counts selection of n0 or n1. "
                  "These marginal intervals accompany the prespecified paired McNemar comparison in the decision table."))

## Public workflow

The example repeats the first frozen zero-inflated confirmation draw. The `simplify_mean=True` argument requests the candidate mean-form procedure explicitly. Discovery sees the fitting rows, while `conformal_interval` receives a separate calibration frame and independent prediction rows. The generating mean is used only for grading; it is not passed to discovery.

In [ ]:
import mboostrs as mb
from benches.study_count_mean_backbone import draw

seed = 6608100
analysis, _, _ = draw("zero_inflated", seed)
evaluation, _, _ = draw("zero_inflated", seed + 100000)
assignment = mb.cv_weights(analysis.height, type="kfold", B=2, seed=seed)[:, 0]
fitting = analysis.filter(pl.Series(assignment > 0))
calibration = analysis.filter(pl.Series(assignment == 0))

In [ ]:
result = mb.discover(fitting, "y", seed=seed, simplify_mean=True)
interval = mb.conformal_interval(result.model, evaluation.drop("y"), calibration=calibration,
                                 seed=seed, level=.9, extrapolation="clip")
result.formula

In [ ]:
recorded = summary["example_record"]
assert (recorded["panel"], recorded["seed"], recorded["method"]) == ("zero_inflated", seed, "shape_discovery")
for key, values in (("lower", interval.lower), ("upper", interval.upper), ("point", interval.fitted_mean)):
    np.testing.assert_allclose(values, recorded["predictions"][key], rtol=0, atol=1e-10)
comparison = [r for r in result.log if r["stage"] == "count mean forms" and "simple_scores" in r]
frame = pl.DataFrame([{"Specification": item, "Mean CV risk": np.mean(comparison[0][item + "_scores"])}
                      for item in ("rich", "simple")]) if comparison else pl.DataFrame({"Specification": ["Not applicable"], "Mean CV risk": [None]})
(GT(frame, rowname_col="Specification").tab_stubhead("Mean form")
 .tab_header(title="Recorded mean-form decision", subtitle="Five fitting folds; lower family risk is better")
 .fmt_number(columns=["Mean CV risk"], decimals=6)
 .tab_source_note("The actual paired decision uses the mean risk difference and its paired fold SE, not the marginal SEs. "
                  "The example matches all 1,000 recorded points and endpoints, "
                  "stored in production_confirmation_summary.json, to 1e-10."))

## Complete evidence

The production source, runner and complete decision analysis were committed at `d7e55ad` before generating seeds 6608100–6608149. Development uses 6607100–6607109; independent test seeds add 100000. All eleven methods and twelve panels are reported in the summary files, including failed supplied presets and unbounded sets. Failed methods have no mean computed only from successes. The development interaction and supplied-family panels share inputs and their repeated preset failure represents one common event.

Native-call replay checks actual score, cutoff and endpoint inputs and results. A separate audit regenerates the analysis/test frames and partitions, replays each mean-form decision, checks the exact positive-grid stopping minimum, and reconstructs competitor calibration scores and endpoints. Hashes identify scripts, loaded Python sources and native source. Measurement-time metadata includes CatBoost and LightGBM versions. Concurrent workflow timing is descriptive; memory was not measured.

In [ ]:
audit = json.loads((directory / "production_confirmation_audit.json").read_text())
frame = pl.DataFrame([{"Check": "Input and partition regeneration", "Result": str(audit["complete_input_partition_regeneration"])},
    {"Check": "Recorded native and endpoint agreement", "Result": str(audit["exact_recorded_native_and_endpoint_agreement"])},
    {"Check": "Unique native calls replayed", "Result": str(sum(audit["native_calls_replayed"].values()))},
    {"Check": "Mean-form decisions replayed", "Result": str(audit["paired_shape_decisions_replayed"])},
    {"Check": "Stopping minima checked", "Result": str(audit["exact_positive_grid_minima_verified"])},
    {"Check": "Competitor intervals reconstructed", "Result": str(audit["competitor_intervals_reconstructed"])},
    {"Check": "Failed records retained", "Result": str(len(audit["failures"]))}])
(GT(frame, rowname_col="Check").tab_stubhead("Evidence check")
 .tab_header(title="Independent confirmation audit", subtitle="All recorded outcomes; exact native-call comparisons")
 .tab_source_note("Sources: production_confirmation_audit.json and benches/verify_count_mean_backbone.py. "
                  "Numerical tolerances in independent metric aggregation do not alter model selection or target gates."))

## Workflow cost

Elapsed time includes fitting, tuning, calibration, prediction and evidence capture, and excludes shared data generation and record serialization. These concurrently measured times are descriptive. Method order and CPU contention prevent interpreting them as an isolated library speed comparison. All fifty elapsed times per method enter the summaries, including failed presets; memory was not measured. Times describe the frozen measurement revision; later optional held-refit integration can change workflow cost.

In [ ]:
timings = [{"Process": r["panel"], "Workflow": labels.get(r["method"], r["method"].replace("_", " ")),
            "Median seconds": r["elapsed_median"], "25th percentile": r["elapsed_q25"],
            "75th percentile": r["elapsed_q75"]}
           for r in rows if r["panel"] in ("original", "zero_inflated")]
(GT(pl.DataFrame(timings), rowname_col="Workflow", groupname_col="Process").tab_stubhead("Procedure")
 .tab_header(title="Observed workflow time", subtitle="Seconds across 50 draws; concurrent runs, descriptive only")
 .fmt_number(columns=["Median seconds", "25th percentile", "75th percentile"], decimals=3)
 .tab_source_note("Interquartile ranges describe measured timings, not confidence intervals. Source: production_confirmation_summary.json, "
                  "whose quartiles use all 50 elapsed times per method. "
                  "All outcomes are retained. Methods execute in the same fixed order within each draw."))

## Reproduction

The source notebook reads saved study results. Run the expensive measurement outside the documentation build, in a fresh output path; the runner refuses to overwrite evidence. Execute the same command for each panel listed in `PANELS`, then run the frozen summary and independent audit.

```bash
PYTHONPATH=python:. python benches/study_count_mean_backbone.py   --phase confirmation --panel zero_inflated   --output docs/book/benchmarks/count_mean_backbone/production_confirmation_zero_inflated.jsonl.xz
PYTHONPATH=python:. python benches/summarize_count_mean_backbone.py   --phase confirmation --prefix production_confirmation
PYTHONPATH=python:. python benches/verify_count_mean_backbone.py --phase confirmation
```

The original protocols, the development and confirmation summary files (`production_development_summary.json`, `production_confirmation_summary.json`) and the validation reports accompany the notebook download. The per-draw archives are assets of the `benchmark-raw-archives-1` GitHub release; `docs/book/benchmarks/raw_archives.json` lists each file with its size, SHA-256 and the command that regenerates it. Research prototypes and rate-ablation evidence remain development evidence, with their original method labels.